## Inspiration

I’ve always been intrigued by how different factors—like a property’s size, location, and age—shape its market value. Real estate is a significant part of our economy, and understanding the drivers of house prices can be both fascinating and practical. That’s why I chose this dataset, which focuses on key attributes such as the number of bedrooms, bathrooms, total square footage, and lot area. By analyzing these features, I hope to see which ones have the most substantial impact on price and thus shed light on how various property characteristics intersect to influence market trends.

This project aligns with my future academic aspirations, where I plan to explore additional property characteristics—like renovation quality, energy efficiency certifications, and detailed home condition ratings—that are often critical drivers of value but are not captured here. Studying these factors can offer deeper insights into how buyers and sellers perceive value in different market conditions. Overall, real estate impacts nearly everyone, whether directly or indirectly, making this exploration a worthwhile step toward more robust, data-driven decision-making in the housing market.

## Stakeholders

Several stakeholders would be interested in hearing the results of my project, including homebuyers, sellers, real estate agents, brokers, investors, urban planners, and policy makers. For potential buyers and sellers, insights into pricing trends can inform decisions about when to enter the market. Real estate agents and brokers would benefit by using this information to better advise their clients and set accurate listing prices. From an investors' perspective, a solid predictive model might help identify undervalued properties. Finally, urban planners and policy makers can apply insights from housing trends to shape zoning laws, infrastructure projects, and affordable housing policies.

## Task and Metrics

For this project, I implemented a regression task because the goal is to predict a continuous outcome—home price—using a range of property features including Bedrooms, Bathrooms, SqFt, City, Province, Year_Built, Type, Garage, and Lot_Area. To evaluate model performance, I chose Root Mean Squared Error (RMSE) and R².

While RMSE was chosen because it penalizes large errors (which are common in expensive housing markets), I also monitored R² to measure the fraction of variance explained. Moreover, Mean Absolute Error (MAE) could be another useful metric to interpret the average magnitude of errors. However, given the typical focus on large errors in real estate contexts, RMSE was best suited for the primary evaluation.

## Data

I found Canadian real estate data from Kaggle [Canadian real estate data](https://www.kaggle.com/datasets/smmmmmmmmmmmm/canada-real-estate?resource=download), which is comprised of 5000 entries across 10 distinct attributes. This dataset simulates diverse real estate properties within major Canadian cities, incorporating essential property features: Price, Bedrooms, Bathrooms, SqFt, City, Province, Year_Built, Type, Garage, and Lot_Area. Below you will find the detaisl about the observations and variables. 

In [9]:
#| echo: false

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.formula.api as smf
from statsmodels.tools.tools import add_constant

from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, Lasso, Ridge, LassoCV, RidgeCV
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split

data = pd.read_csv('ca_real_estate.csv')

num_rows, num_cols = data.shape
print(f"\nNumber of observations (rows): {num_rows}")
print(f"Number of variables (columns): {num_cols}")

continuous_vars = data.select_dtypes(include=[np.number]).columns.tolist()
categorical_vars = data.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"\nNumber of continuous variables: {len(continuous_vars)}")
print(f"Number of categorical variables: {len(categorical_vars)}")

missing_values = data.isnull().sum()
print("\nMissing Values Summary:")
print(missing_values)
print('\nBecause all values are found, cleaning/imputing is not necessary')


Number of observations (rows): 5000
Number of variables (columns): 10

Number of continuous variables: 7
Number of categorical variables: 3

Missing Values Summary:
Price         0
Bedrooms      0
Bathrooms     0
SqFt          0
City          0
Province      0
Year_Built    0
Type          0
Garage        0
Lot_Area      0
dtype: int64

Because all values are found, cleaning/imputing is not necessary


In [10]:
#| echo: false

corr_matrix = data[continuous_vars].corr()
print("\nCorrelation Matrix (Continuous Variables):")
corr_matrix


Correlation Matrix (Continuous Variables):


,Price,Bedrooms,Bathrooms,SqFt,Year_Built,Garage,Lot_Area
Price,1.000000,-0.032169,-0.002194,-0.030875,0.009489,0.000591,-0.002460
Bedrooms,-0.032169,1.000000,0.018430,-0.004158,-0.001657,0.006680,-0.010404
Bathrooms,-0.002194,0.018430,1.000000,0.006462,0.009065,0.002102,-0.008651
SqFt,-0.030875,-0.004158,0.006462,1.000000,-0.024400,-0.006923,-0.001564
Year_Built,0.009489,-0.001657,0.009065,-0.024400,1.000000,0.012056,-0.004763
Garage,0.000591,0.006680,0.002102,-0.006923,0.012056,1.000000,-0.014666
Lot_Area,-0.002460,-0.010404,-0.008651,-0.001564,-0.004763,-0.014666,1.000000


It is important to note that this data appears to have low correlations among key variables, which might indicate artificial or incomplete generation processes. This could limit the model’s ability to replicate real-world trends in Canadian real estate accurately.

## Prediction

In [13]:
#| echo: false

y = data['Price']
X = data[['Bedrooms', 'Bathrooms', 'SqFt', 'City', 'Province', 'Year_Built', 'Type', 'Garage', 'Lot_Area']]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

I began by creating a simple unregularized linear regression model, the variable most traditionally associated with price (SqFt). I then proceeded to add Bedrooms and Bathrooms, which often correlate with home size and livability, followed by Year_Built and Lot_Area as structural and land-related factors, and finally adding the categorical variables (City, Province, and Type) and Garage.

To make matters easy, I used a for loop in order to perform a stepwise addition of predictors. The training and test RMSE for each model were recorded as follows:

In [15]:
#| echo: false

def compute_rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def get_train_test_rmse(formula, train_df, test_df, y_col='Price'):
    model = smf.ols(formula=formula, data=train_df).fit()
    y_train_pred = model.predict(train_df)
    y_test_pred  = model.predict(test_df)
    rmse_train = compute_rmse(train_df[y_col], y_train_pred)
    rmse_test  = compute_rmse(test_df[y_col],  y_test_pred)
    return rmse_train, rmse_test

train_df = X_train.copy()
train_df['Price'] = y_train

test_df = X_test.copy()
test_df['Price'] = y_test

predictors_in_order = [
    ['SqFt'],  
    ['SqFt', 'Bedrooms'],
    ['SqFt', 'Bedrooms', 'Bathrooms'],
    ['SqFt', 'Bedrooms', 'Bathrooms', 'Year_Built'],
    ['SqFt', 'Bedrooms', 'Bathrooms', 'Year_Built', 'Lot_Area'],
    ['SqFt', 'Bedrooms', 'Bathrooms', 'Year_Built', 'Lot_Area', 'C(City)'],
    ['SqFt', 'Bedrooms', 'Bathrooms', 'Year_Built', 'Lot_Area', 'C(City)', 'C(Province)'],
    ['SqFt', 'Bedrooms', 'Bathrooms', 'Year_Built', 'Lot_Area', 'C(City)', 'C(Province)', 'C(Type)'],
    ['SqFt', 'Bedrooms', 'Bathrooms', 'Year_Built', 'Lot_Area', 'C(City)', 'C(Province)',
     'C(Type)', 'Garage']
]

results = []
for step_preds in predictors_in_order:
    formula_str = "Price ~ " + " + ".join(step_preds)
    rmse_train, rmse_test = get_train_test_rmse(formula_str, train_df, test_df, 'Price')
    results.append({
        'Predictors': step_preds,
        'Train_RMSE': rmse_train,
        'Test_RMSE': rmse_test
    })

results_df = pd.DataFrame(results)
print("\n=== Stepwise addition of predictors (unregularized) ===")
results_df.head(10)


=== Stepwise addition of predictors (unregularized) ===


,Predictors,Train_RMSE,Test_RMSE
0,[SqFt],259519.721356,256901.538533
1,"[SqFt, Bedrooms]",259278.861555,257266.070031
2,"[SqFt, Bedrooms, Bathrooms]",259278.802886,257267.964925
3,"[SqFt, Bedrooms, Bathrooms, Year_Built]",259269.709495,257256.716240
4,"[SqFt, Bedrooms, Bathrooms, Year_Built, Lot_Area]",259245.450866,257426.132401
5,"[SqFt, Bedrooms, Bathrooms, Year_Built, Lot_Ar...",259198.100609,257543.209325
6,"[SqFt, Bedrooms, Bathrooms, Year_Built, Lot_Ar...",259056.053642,257478.854179
7,"[SqFt, Bedrooms, Bathrooms, Year_Built, Lot_Ar...",258848.195340,257929.608778
8,"[SqFt, Bedrooms, Bathrooms, Year_Built, Lot_Ar...",258847.602698,257934.279403


From the table, the training RMSE steadily decreases as more predictors are added (from about 259,520 down to around 258,848), but the test RMSE initially rises and then remains at a higher level than it started. In other words, while the model fits the training data more closely with each added term, its performance on unseen data actually worsens relative to the simpler model. This is a clear indication of overfitting: by adding more features, the model is capturing nuances (and possibly noise) in the training set that do not generalize well to the test set. Because of the overfitting, I will use regularization. 

In [17]:
#| echo: false

num_cols = ['Bedrooms', 'Bathrooms', 'SqFt', 'Year_Built', 'Garage', 'Lot_Area']
cat_cols = ['City', 'Province', 'Type'] 

# OHE
X_train_num = X_train[num_cols]
X_test_num  = X_test[num_cols]

X_train_cat = pd.get_dummies(X_train[cat_cols], drop_first=True)
X_test_cat  = pd.get_dummies(X_test[cat_cols], drop_first=True)

X_test_cat = X_test_cat.reindex(columns=X_train_cat.columns, fill_value=0)

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train_num) 
X_test_poly  = poly.transform(X_test_num)

poly_feature_names = poly.get_feature_names_out(num_cols)

X_train_expanded = np.concatenate([X_train_poly, X_train_cat.values], axis=1)
X_test_expanded  = np.concatenate([X_test_poly,  X_test_cat.values ], axis=1)

scaler = StandardScaler()
scaler.fit(X_train_expanded)
X_train_scaled = scaler.transform(X_train_expanded)
X_test_scaled  = scaler.transform(X_test_expanded)

In [18]:
#| echo: false

# Perform ridgeCV to see if better
ridge_cv = RidgeCV(alphas=10**np.linspace(5, -2, 100)*0.5, cv=5)
ridge_cv.fit(X_train_scaled, y_train)
print("Best Ridge Alpha:", ridge_cv.alpha_)
ridge_test_rmse = np.sqrt(mean_squared_error(y_test, ridge_cv.predict(X_test_scaled)))
print("Ridge Test RMSE:", ridge_test_rmse)

# Perform Lasscv to see if better
alphas_to_try = 10**np.linspace(5, -2, 200) * 0.5 
model_cv = LassoCV(alphas=alphas_to_try, cv=5, max_iter=200_000)
model_cv.fit(X_train_scaled, y_train)
print("\nBest Lasso Alpha:", model_cv.alpha_)
y_test_pred_cv = model_cv.predict(X_test_scaled)
test_rmse_cv = np.sqrt(mean_squared_error(y_test, y_test_pred_cv))
print("Test RMSE (with best alpha using Lasso) =", test_rmse_cv)

Best Ridge Alpha: 15996.335688986923
Ridge Test RMSE: 257332.55500989687

Best Lasso Alpha: 7761.12678713524
Test RMSE (with best alpha using Lasso) = 257214.5198652233


After running both LassoCV and RidgeCV, I found that the models produced very similar results, with Lasso performing slightly better. The best alpha for Lasso was 7761.13, yielding a test RMSE of 257214.52, while Ridge’s best alpha was 15996.34, resulting in a slightly higher test RMSE of 257332.56. Although the difference is small, Lasso’s lower RMSE suggests it made marginally better predictions; therefore, Lasso is the better choice. As noted previously, low correlations among predictors may be contributing to the model's limited predictive power.

In [20]:
#| echo: false

all_cat_dummies = list(X_train_cat.columns)  # these come after the poly numeric features
all_feature_names = list(poly_feature_names) + all_cat_dummies

coef_array = model_cv.coef_
coef_df = pd.DataFrame({
    'feature': all_feature_names,
    'coefficient': coef_array
})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
coef_df.sort_values('abs_coef', ascending=False, inplace=True)

print("\n--- Top 5 Largest Coefficients (by absolute value) ---")
print(coef_df.head(5)[['feature','coefficient']])


--- Top 5 Largest Coefficients (by absolute value) ---
              feature  coefficient
8       Bedrooms SqFt -5585.853717
35         Type_House -1852.924357
27      City_Montreal     0.000000
21       Year_Built^2     0.000000
22  Year_Built Garage     0.000000


Among all features analyzed, certain polynomial and interaction terms stood out as most influential. What struck out to me the most was the interaction term ‘Bedrooms × SqFt’ because it was negative. One plausible explanation is that very large homes with many bedrooms in this dataset might be in more remote areas or might lack premium finishes, resulting in lower prices compared to smaller upscale homes. Thus, the model shows ‘diminishing returns’ on adding both size and bedroom count simultaneously.
This results underscore the importance of modeling nonlinear relationships rather than relying solely on linear terms.

In [22]:
#| echo: false

print("\n--- Bottom 5 (smallest absolute value) ---")
print(coef_df.tail(5)[['feature','coefficient']])


--- Bottom 5 (smallest absolute value) ---
                 feature  coefficient
13        Bathrooms SqFt         -0.0
14  Bathrooms Year_Built          0.0
15      Bathrooms Garage          0.0
16    Bathrooms Lot_Area         -0.0
18       SqFt Year_Built         -0.0


These coefficients indicate that interaction terms involving bathrooms with other features like square footage, year built, garage, and lot area—as well as the interaction between square footage and year built—do not significantly contribute to predicting home prices. This suggests that these specific interactions provide minimal explanatory power and can likely be excluded without sacrificing model accuracy.

In [24]:
#| echo: false

zero_mask = (coef_df['coefficient'] == 0.0)
zeros_df = coef_df[zero_mask]
if len(zeros_df) > 0:
    print(f"\nNumber of zero-coeff terms: {len(zeros_df)}")
    print(zeros_df[['feature','coefficient']])


Number of zero-coeff terms: 34
                 feature  coefficient
27         City_Montreal          0.0
21          Year_Built^2          0.0
22     Year_Built Garage          0.0
23   Year_Built Lot_Area         -0.0
24              Garage^2          0.0
25       Garage Lot_Area          0.0
26            Lot_Area^2         -0.0
28           City_Ottawa         -0.0
19           SqFt Garage          0.0
29          City_Toronto          0.0
30        City_Vancouver          0.0
31           Province_BC          0.0
32           Province_ON          0.0
33           Province_QC         -0.0
34            Type_Condo          0.0
20         SqFt Lot_Area         -0.0
0               Bedrooms         -0.0
1              Bathrooms         -0.0
9    Bedrooms Year_Built         -0.0
2                   SqFt         -0.0
3             Year_Built          0.0
4                 Garage          0.0
5               Lot_Area         -0.0
6             Bedrooms^2         -0.0
7     Bedrooms Bat

Interestingly, many features originally thought to be important individually had minimal or no independent influence. Features such as SqFt, Bathrooms, Garage, Year_Built, and Lot Area had coefficients reduced to zero, indicating their limited standalone predictive value once interactions and polynomial terms were included. Likewise, dummy variables for specific cities generally had no significant effect, reinforcing that property prices were less sensitive to specific city locations within the dataset once other characteristics were accounted for.

Overall, these results suggest that future modeling efforts should prioritize capturing nonlinear relationships and interaction effects rather than emphasizing individual property attributes alone. It also highlights the importance of understanding regional market dynamics and property type distinctions when predicting housing prices.

## Inference

In [28]:
#| echo: false

final_formula = """
Price ~ SqFt
       + Bedrooms
       + Bathrooms
       + Year_Built
       + C(Type)
       + I(Bedrooms**2)
       + Bedrooms:Bathrooms
       + Bedrooms:SqFt
       + C(City)
       + C(Province)
"""

final_model = smf.ols(final_formula, data=train_df).fit()
print("\n--- FINAL MODEL SUMMARY (statsmodels) ---")
print(final_model.summary())

final_preds = final_model.predict(test_df)
final_rmse = np.sqrt(mean_squared_error(test_df['Price'], final_preds))
print("\nFinal model test RMSE =", final_rmse)


--- FINAL MODEL SUMMARY (statsmodels) ---
                            OLS Regression Results                            
Dep. Variable:                  Price   R-squared:                       0.007
Model:                            OLS   Adj. R-squared:                  0.003
Method:                 Least Squares   F-statistic:                     1.691
Date:                Fri, 21 Mar 2025   Prob (F-statistic):             0.0414
Time:                        23:17:22   Log-Likelihood:                -55529.
No. Observations:                4000   AIC:                         1.111e+05
Df Residuals:                    3983   BIC:                         1.112e+05
Df Model:                          16                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------

For my final inference model, I selected the strongest or most interesting terms as guided by Lasso’s non-zero coefficients and domain knowledge. Specifically, since Lasso indicated that most coefficients were zero, I retained Bedrooms, Bathrooms, SqFt, City, Province, and Type, plus a few interactions (like Bedrooms × Bathrooms) that showed consistent effects in the regularized model.

Among these predictors, only the property type "House" was statistically significant (p = 0.015), with a negative coefficient of approximately -$24,480, indicating that houses in this dataset, all else equal, are priced lower than the baseline property type. This somewhat surprising result might reflect specific characteristics or market dynamics within the sampled properties. 

Additionally, the interaction between bedrooms and bathrooms showed a marginally significant positive coefficient of about $6,084 (p = 0.082), suggesting that homes with a balanced mix of bedrooms and bathrooms might be more appealing to buyers and thus command higher prices. However, other predictors, including city and province indicators, square footage, year built, the quadratic term for bedrooms, and the interaction between bedrooms and square footage, did not exhibit statistically significant effects (all p-values > 0.05), indicating limited explanatory power for these variables in predicting home prices.

Regarding reliability, it's critical to acknowledge the extremely low R-squared value (approximately 0.7%), meaning this model explains less than 1% of the variation in prices. This indicates that these predictors explain very little of the variance in housing prices in this dataset. This underscores that many critical price drivers (e.g., property condition, renovation history, neighborhood-specific factors, etc.) are absent from our data. Furthermore, the high condition number (8.56e+05) signals potential multicollinearity issues, possibly inflating standard errors and obscuring significant relationships among predictors. The model's predictive performance, with a large RMSE of approximately $258,568, reinforces that substantial unexplained variation remains.

## Recommendation to Stakeholders

Given these findings, stakeholders—such as real estate agents, property developers, or homeowners—should consider specific strategic actions. First, they should carefully evaluate why houses appear undervalued relative to other property types, potentially conducting targeted market research or qualitative assessments to understand buyer perceptions. Second, developers and renovators should prioritize balanced enhancements to both bedrooms and bathrooms, rather than solely adding bedrooms, to maximize property appeal and market value.

Because my model’s explanatory power (R² ≈ 0.007) is very limited, stakeholders should be cautious when applying these results in practice. Crucial data such as neighborhood crime rates, school quality, property condition, and local economic indicators were not included in the dataset. To improve accuracy, I recommend collecting more granular location data (e.g., neighborhood or postal code), detailed property condition scores, renovation histories, and broader economic factors. Implementing a regular check for multicollinearity (e.g., by computing Variance Inflation Factors) and potentially using a log transform of Price are additional steps that could enhance the model’s reliability.

## Conclusion

In summary, my analysis reveals that adding bedrooms alone may have a negligible or negative impact on Price unless paired with additional bathrooms, underscoring the synergy between these two features. However, the model’s overall predictive and explanatory power remains minimal (R² near 0.7%, RMSE around 259k), meaning many other crucial factors are not captured by the data. Future efforts should focus on gathering more detailed location information and property-specific attributes to meaningfully improve predictive accuracy.